mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


installing libaries

In [ ]:
!pip install tensorflow opencv-python matplotlib

Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

define dataset path

In [ ]:
dataset_path = "/content/drive/MyDrive/FaceMaskDataset"

Create Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 8000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


Load MobileNetV2

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Freeze Base Layers

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

Build Custom Classifier

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

predictions = Dense(
    2,
    activation='softmax'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=predictions
)

complie model

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model summary

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,210 (9.24 MB)

 Trainable params: 164,226 (641.51 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)

Training samples: 8000
Validation samples: 2000


Train Model

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1535s 6s/step - accuracy: 0.9744 - loss: 0.0672 - val_accuracy: 0.9650 - val_loss: 0.0934
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 167s 670ms/step - accuracy: 0.9877 - loss: 0.0335 - val_accuracy: 0.9880 - val_loss: 0.0326
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 162s 647ms/step - accuracy: 0.9898 - loss: 0.0302 - val_accuracy: 0.9870 - val_loss: 0.0347
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 161s 646ms/step - accuracy: 0.9937 - loss: 0.0190 - val_accuracy: 0.9735 - val_loss: 0.0673
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 170s 680ms/step - accuracy: 0.9927 - loss: 0.0226 - val_accuracy: 0.9895 - val_loss: 0.0281
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 163s 652ms/step - accuracy: 0.9923 - loss: 0.0192 - val_accuracy: 0.9900 - val_loss: 0.0308
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 168s 672ms/step - accuracy: 0.9941 - loss: 0.0160 - val_accuracy: 0.9930 - val_loss: 0.0268
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 163s 652ms/step - accuracy: 0.9935 - l

Evaluate Accuracy

In [ ]:
loss, accuracy = model.evaluate(val_generator)

print("Validation Accuracy:", accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 32s 504ms/step - accuracy: 0.9870 - loss: 0.0380
Validation Accuracy: 0.9869999885559082


Plot Results
1. Accuracy vs Epochs Graph Code

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title('Accuracy vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

plt.legend(loc='lower right')

# Save the graph
plt.savefig("accuracy_vs_epochs.png")

plt.show()

SyntaxError: invalid syntax (1093136855.py, line 18)

confusion matriz

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

val_generator.reset()

predictions = model.predict(val_generator)

predicted_classes = np.argmax(predictions, axis=1)

cm = confusion_matrix(
    val_generator.classes,
    predicted_classes
)

print(cm)

63/63 ━━━━━━━━━━━━━━━━━━━━ 51s 647ms/step
[[494 506]
 [493 507]]


SAVE MODEL

In [ ]:
model.save("mask_detector.h5")

DOWNLOAD MODEL

In [ ]:
from google.colab import files

files.download("mask_detector.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>